# Two Sudoku Solvers: Implication / SAT Solver vs. Recursion

I wrote two solvers for Sudoku puzzles and compared them against each other (I used Claude Code to generate the benchmarking harness). Solver A turns the puzzles into one big, logical expression (Applying what I learned CSE 311: Foundations of Computing I) and solves that. Solver B uses recursive backtracking which I learned in CSE 143 (Computer Programming II). Results are discussed in [Section 8](#takeaways).

*by Abhinav Pandey*

## 0. Grid helpers

The grid is a 9x9 list of lists and `0` means blank.

In [ ]:
import time
import tracemalloc


def parse_grid(text):
    # '.' and '0' both mean blank, I allow either
    cells = []
    for ch in text:
        if ch in "0123456789.":
            cells.append(ch)

    grid = []
    for r in range(9):
        row = []
        for c in range(9):
            ch = cells[r * 9 + c]
            if ch == "." or ch == "0":
                row.append(0)
            else:
                row.append(int(ch))
        grid.append(row)
    return grid


def show(grid):
    # prints the board with the 3x3 lines so I can actually read it
    if grid is None:
        return "<no solution>"

    lines = []
    for r in range(9):
        if r == 3 or r == 6:
            lines.append("------+-------+------")
        line = ""
        for c in range(9):
            if c == 3 or c == 6:
                line = line + "| "
            if grid[r][c] == 0:
                line = line + ". "
            else:
                line = line + str(grid[r][c]) + " "
        lines.append(line)
    return "\n".join(lines)


def all_units():
    # the 27 units: 9 rows, 9 cols, 9 boxes. every rule is about these, so
    # I build them once
    groups = []

    for r in range(9):
        row = []
        for c in range(9):
            row.append((r, c))
        groups.append(row)

    for c in range(9):
        col = []
        for r in range(9):
            col.append((r, c))
        groups.append(col)

    for br in range(3):
        for bc in range(3):
            box = []
            for dr in range(3):
                for dc in range(3):
                    box.append((br * 3 + dr, bc * 3 + dc))
            groups.append(box)

    return groups

## 1. Sudoku as logic: Solver A (Part I)

The trick is to give every (cell, value) pair its own proposition:

$$P(r,c,v) \;=\; \text{“cell } (r,c) \text{ has digit } v\text{.”}$$

That's 729 variables. Then the rules turn into CNF clauses:

- at least one value per cell
- at most one value per cell
- every digit at least once per row / col / box
- every digit at most once per unit — technically redundant, but propagation gets more information out of it
- the givens, as one-literal clauses

In [ ]:
def var(r, c, v):
    # (r, c, v) squashed into one number in 1..729
    return r * 81 + c * 9 + (v - 1) + 1


def at_most_one(lits):
    # CNF for "at most one of these is true": (NOT a OR NOT b) for every pair
    clauses = []
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            clauses.append([-lits[i], -lits[j]])
    return clauses


def encode_sudoku(grid):
    # the rules written as CNF: a list of clauses, each one a list of literals
    # (positive = the var, negative = NOT the var)
    clauses = []
    units = all_units()

    # rules 1-2: every cell gets at least one value, and at most one
    for r in range(9):
        for c in range(9):
            cell_vars = []
            for v in range(1, 10):
                cell_vars.append(var(r, c, v))

            clauses.append(cell_vars)
            for clause in at_most_one(cell_vars):
                clauses.append(clause)

    # rules 3-4: every digit shows up at least once and at most once in each
    # row/col/box. rule 4 is redundant but it gives propagation way more to
    # work with, so the solver barely has to guess
    for v in range(1, 10):
        for unit in units:
            unit_vars = []
            for cell in unit:
                unit_vars.append(var(cell[0], cell[1], v))

            clauses.append(unit_vars)
            for clause in at_most_one(unit_vars):
                clauses.append(clause)

    # rule 5: the givens, as one-literal clauses
    for r in range(9):
        for c in range(9):
            if grid[r][c] != 0:
                clauses.append([var(r, c, grid[r][c])])

    return clauses

## 2. DPLL: Unit propagation: Solver A (Part II)

The most interesting part of the code: If every literal in a clause but one is false, the last one is forced:

$$(\lnot\ell_1\land\lnot\ell_2)\rightarrow\ell_3.$$

That's just an implication, and so we can use modus ponens. So DPLL is:

- **propagate** — fire forced implications until nothing changes, or some clause has every literal false (a contradiction)
- **decide** — when that stalls, guess a variable
- **backtrack** — if the guess blows up, undo it and try the other value

I keep an index from each literal to the clauses holding it, so propagation only rechecks the clauses a new assignment could actually break.

In [ ]:
def literal_value(lit, value):
    # True / False / None for this literal, with the negation applied
    v = value[abs(lit)]
    if v is None:
        return None
    if lit > 0:
        return v
    return not v


def build_index(clauses):
    # literal -> which clauses contain it
    index = {}
    for i in range(len(clauses)):
        for lit in clauses[i]:
            if lit not in index:
                index[lit] = []
            index[lit].append(i)
    return index


def propagate(queue, clauses, index, value, trail, stats):
    # fire forced implications until the queue runs dry. False = contradiction
    while len(queue) > 0:
        lit = queue.pop(0)
        state = literal_value(lit, value)

        if state is True:
            continue  # already true, skip it
        if state is False:
            stats["conflicts"] += 1  # asked to make it true, it's false
            return False

        value[abs(lit)] = lit > 0  # commit it
        trail.append(abs(lit))  # remember the order so I can undo
        stats["propagations"] += 1

        # lit is true now, so -lit went false. only clauses holding -lit can break
        for i in index.get(-lit, []):
            clause = clauses[i]
            unknown = []
            satisfied = False
            for other in clause:
                s = literal_value(other, value)
                if s is True:
                    satisfied = True  # something's true, clause is fine
                    break
                if s is None:
                    unknown.append(other)

            if satisfied:
                continue
            if len(unknown) == 0:
                stats["conflicts"] += 1  # everything false, dead
                return False
            if len(unknown) == 1:
                queue.append(unknown[0])  # one left, so it's forced

    return True


def undo_to(mark, value, trail):
    # throw away every assignment made after mark
    while len(trail) > mark:
        v = trail.pop()
        value[v] = None


def search(clauses, index, value, trail, stats):
    # grab the first unknown variable and try it both ways
    v0 = 0
    for v in range(1, len(value)):
        if value[v] is None:
            v0 = v
            break

    if v0 == 0:
        return True  # nothing unknown and no conflict, so SAT

    stats["decisions"] += 1
    for guess in [v0, -v0]:
        mark = len(trail)  # bookmark to roll back to
        if propagate([guess], clauses, index, value, trail, stats):
            if search(clauses, index, value, trail, stats):
                return True
        undo_to(mark, value, trail)  # branch died, wipe it
    return False


def solve_sat(grid):
    # encode, solve, then read the true P(r, c, v)'s back into a grid
    clauses = encode_sudoku(grid)
    index = build_index(clauses)
    value = [None] * 730  # value[v] is True / False / None
    trail = []
    stats = {"variables": 729, "clauses": len(clauses),
             "decisions": 0, "propagations": 0, "conflicts": 0}

    # the givens are the size-1 clauses, so propagate those first
    start = []
    for clause in clauses:
        if len(clause) == 1:
            start.append(clause[0])

    if not propagate(start, clauses, index, value, trail, stats):
        return None, stats  # the givens contradict each other
    if not search(clauses, index, value, trail, stats):
        return None, stats

    out = []
    for r in range(9):
        row = [0] * 9
        for c in range(9):
            for v in range(1, 10):
                if value[var(r, c, v)]:
                    row[c] = v
        out.append(row)
    return out, stats

## 3. The recursive approach (Solver B)

Same rules, but checked with an `is_ok(r, c, v)` function instead of a formula. The biggest difference is this solver guesses first and only finds out the guess was bad much later when some cells have no legal digit. Solver A deduces everything it can before it ever guesses.

In [ ]:
def is_ok(grid, r, c, v):
    # can v go at (r, c)? check the row, the column, the box
    for i in range(9):
        if grid[r][i] == v or grid[i][c] == v:
            return False

    br = 3 * (r // 3)
    bc = 3 * (c // 3)
    for dr in range(3):
        for dc in range(3):
            if grid[br + dr][bc + dc] == v:
                return False
    return True


def find_empty(grid):
    # first blank, left to right / top to bottom. None if the grid is full
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:
                return r, c
    return None


def backtrack(grid, stats):
    stats["calls"] += 1  # counting work for the comparison
    cell = find_empty(grid)
    if cell is None:
        return True  # no blanks left, so it's solved

    r = cell[0]
    c = cell[1]
    for v in range(1, 10):
        if is_ok(grid, r, c, v):
            grid[r][c] = v  # take the guess
            stats["assignments"] += 1
            if backtrack(grid, stats):
                return True
            grid[r][c] = 0  # didn't work, pull it back out
            stats["backtracks"] += 1
    return False  # nothing fits here, dead end


def solve_backtracking(grid):
    # returns (grid, stats)
    work = []
    for row in grid:  # copy so I don't wreck the input
        work.append(list(row))

    stats = {"calls": 0, "assignments": 0, "backtracks": 0}
    if backtrack(work, stats):
        return work, stats
    return None, stats

## 4. Benchmark harness-Generated by Claude Code

`perf_counter` for time, `tracemalloc` peak for memory. For SAT, the code times the encoding *and* the solve, since building the formula is part of what it costs. Additionally, also count the work each side does, and check both solvers land on the same grid.

In [ ]:
def run(solve_fn, grid, repeats=1):
    # run it a few times, give back median time, worst memory peak, last result
    times = []
    peak = 0
    result = None
    for i in range(repeats):
        tracemalloc.start()
        start = time.perf_counter()
        result = solve_fn(grid)
        times.append(time.perf_counter() - start)
        current, this_peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        if this_peak > peak:
            peak = this_peak
    times.sort()
    return times[len(times) // 2], peak, result


def fmt_bytes(n):
    if n < 1024:
        return str(round(n, 1)) + " B"
    if n < 1024 * 1024:
        return str(round(n / 1024, 1)) + " KB"
    return str(round(n / 1024 / 1024, 1)) + " MB"


def print_row(label, left, right):
    # one row of the table. pass "-" when a metric doesn't apply
    print(str(label).ljust(24) + str(left).rjust(16) + str(right).rjust(18))


def compare(grid, label=""):
    # run both solvers on one puzzle and dump the numbers side by side
    if label != "":
        print("\n" + "#" * 58)
        print("#  " + label)
        print("#" * 58)
    print("\nPuzzle:")
    print(show(grid))

    t_sat, mem_sat, sat_out = run(solve_sat, grid)
    t_bt, mem_bt, bt_out = run(solve_backtracking, grid)
    sol_sat = sat_out[0]
    stat_sat = sat_out[1]
    sol_bt = bt_out[0]
    stat_bt = bt_out[1]

    # a real puzzle has one solution, so both had better agree
    print("\nSolution:")
    print(show(sol_sat))
    print("\nBoth methods agree:", sol_sat == sol_bt)

    print("\n" + "=" * 58)
    print_row("Metric", "SAT", "Backtracking")
    print("-" * 58)
    print_row("time (ms)", round(t_sat * 1000, 2), round(t_bt * 1000, 2))
    print_row("peak memory", fmt_bytes(mem_sat), fmt_bytes(mem_bt))
    print("-" * 58)

    print("Work performed:")
    print_row("  branch decisions", stat_sat["decisions"], "-")
    print_row("  implications fired", stat_sat["propagations"], "-")
    print_row("  contradictions", stat_sat["conflicts"], "-")
    print_row("  recursive calls", "-", stat_bt["calls"])
    print_row("  trial assignments", "-", stat_bt["assignments"])
    print_row("  backtracks", "-", stat_bt["backtracks"])
    print("-" * 58)

    print("Formula size (SAT only):")
    print_row("  propositional vars", stat_sat["variables"], "-")
    print_row("  CNF clauses", stat_sat["clauses"], "-")
    print("=" * 58)


def summary(puzzles):
    # one table: time + peak memory for both solvers on every puzzle
    print("\n" + "=" * 58)
    print("RUNNING TIME & MEMORY COMPARISON")
    print("=" * 58)
    print_row("Puzzle", "time (ms)", "peak mem")
    print("-" * 58)

    times = {}
    mems = {}
    for name, grid in puzzles:
        for method, fn in (("SAT", solve_sat), ("Backtracking", solve_backtracking)):
            t, mem, result = run(fn, grid, 5)
            times[(name, method)] = t
            mems[(name, method)] = mem
            print_row("  " + name + " / " + method, round(t * 1000, 2),
                      fmt_bytes(mem))
        print("-" * 58)

    print("Speed:")
    for name, grid in puzzles:
        t_sat = times[(name, "SAT")]
        t_bt = times[(name, "Backtracking")]
        if t_sat < t_bt:
            print("  " + name + ": SAT is " + str(round(t_bt / t_sat, 1)) + "x faster")
        else:
            print("  " + name + ": backtracking is " + str(round(t_sat / t_bt, 1)) + "x faster")

    # memory barely moves between puzzles, so I just use the first one
    first = puzzles[0][0]
    ratio = mems[(first, "SAT")] / mems[(first, "Backtracking")]
    print("Memory:")
    print("  SAT uses about " + str(round(ratio)) + "x more memory than backtracking")
    print("=" * 58)


# normal hard-newspaper puzzle, basically no guessing needed
CLASSIC = ("53..7...."
           "6..195..."
           ".98....6."
           "8...6...3"
           "4..8.3..1"
           "7...2...6"
           ".6....28."
           "...419..5"
           "....8..79")

# Arto Inkala's "world's hardest". this one makes backtracking sweat
HARD = ("8........"
        "..36....."
        ".7..9.2.."
        ".5...7..."
        "....457.."
        "...1...3."
        "..1....68"
        "..85...1."
        ".9....4..")

## 5. Puzzle 1: Classic puzzle
Picked by Claude Code.

In [ ]:
compare(parse_grid(CLASSIC), 'Puzzle 1  --  classic')

## 6.Puzzle 2: Inkala's 'hardest' puzzle

Picked by Claude Code.

In [ ]:
compare(parse_grid(HARD), 'Puzzle 2  --  Inkala hardest')

## 7. Time & memory, head to head-Generated by Claude Code

Median of 5 runs because wall-clock timing is noisy, and the `tracemalloc` peak for memory.

In [ ]:
summary([('classic', parse_grid(CLASSIC)), ('hardest', parse_grid(HARD))])

<a id='takeaways'></a>
## 8. Takeaways

The thing that surprised me the most was that on Puzzle 1 (easier puzzle), the SAT guessed **zero** times. The givens plus the CNF rules forced every cell, so propagation finished the board by itself. Backtracking on the other hand, had to brute force a few thousand dead ends (~4k) and finished roughly 2x slower than the SAT solver; additionally, I was surprised that even when the SAT had to build a ~12,000 clause equation, it outperformed the Backtracking on time. However, the Inkala puzzle changed that. Backtracking blows up to ~50k recursive calls and gets slow, while SAT only guesses about **27** times because propagation keeps chopping the tree smaller. One caveat to that was the SAT used a lot of memory: a few MB for the formula vs. a few KB for one grid and the call stack. However, what I have learned from CSE 332 (Data Structures and Parallelism) is the worst-case for both puzzles is exponential time as SAT is NP-complete and the recursive approach is bounded by $O(9^n)$ where n is the number of unfilled cells, so neither is magical; however, the SAT solver has a more constant factor.